# PG-LIF — P1 Diagnostics: why does PG-LIF fail on real SHD?
**Context.** PG-LIF v2 reaches 12% on SHD while TC-LIF reaches 75%, yet the byte-identical cell wins on every SHD-like synthetic benchmark tried offline (dense inputs, silent tails, speaker-like warp — up to 82–91%). Only a broadband onset burst reproduced part of the damage, and it hurt ALIF equally, implicating the adaptation pathway. Synthetic proxies are exhausted; this notebook measures the real thing.

**Part A — telemetry (~4 min):** trains the exact v2 PG-LIF for 5 epochs on real SHD while logging train *and* test accuracy, plateau/event/spike/adaptation statistics, surrogate liveness (fraction of steps where the surrogate gradient is nonzero), and per-group gradient norms. These numbers distinguish: dead gradients vs. state saturation vs. representational collapse vs. plain optimization difficulty.

**Part B — bracketing sweep (~15 min):** five PG-LIF variants, 5 epochs each, train+test accuracy: baseline; **no adaptation (β = 0)** — the prime suspect, since TC-LIF has no adaptation and is the strongest model; small plateau gain (κ init 0.25); fast plateau (τp = 15 steps); high dendritic threshold (θd = 2).

Everything is saved to `My Drive/PG_LIF/P1_results/diagnostics_<stamp>/`. GPU runtime required; SHD is reused from the Drive cache.

In [ ]:
import os, json, time, math
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/My Drive'
    if not os.path.isdir(ROOT): ROOT = '/content/drive/MyDrive'
    BASE = os.path.join(ROOT, 'PG_LIF')
except Exception:
    BASE = './PG_LIF'
DATA = os.path.join(BASE, 'data', 'SHD')
OUT = os.path.join(BASE, 'P1_results', 'diagnostics_' + time.strftime('%Y%m%d_%H%M%S'))
os.makedirs(OUT, exist_ok=True)
T_BINS, MAX_TIME, N_IN, N_OUT, HIDDEN, BATCH, LR = 100, 1.4, 700, 20, 128, 64, 5e-4
print('Diagnostics folder:', OUT)

In [ ]:
import numpy as np, h5py, torch, torch.nn as nn
def load_split(fname):
    with h5py.File(os.path.join(DATA, fname), 'r') as f:
        return ([np.array(t) for t in f['spikes']['times']],
                [np.array(u) for u in f['spikes']['units']],
                np.array(f['labels'], dtype=np.int64))
TR = load_split('shd_train.h5'); TE = load_split('shd_test.h5')
def batches(split, batch_size, shuffle, device='cpu'):
    times, units, labels = split
    idx = np.random.permutation(len(labels)) if shuffle else np.arange(len(labels))
    for b0 in range(0, len(idx), batch_size):
        sel = idx[b0:b0+batch_size]
        x = torch.zeros(len(sel), T_BINS, N_IN)
        for i, j in enumerate(sel):
            tt = times[j]; uu = units[j]; keep = tt < MAX_TIME
            tb = np.clip((tt[keep] / MAX_TIME * T_BINS).astype(int), 0, T_BINS-1)
            x[i, tb, uu[keep]] = 1.0
        yield x.to(device), torch.as_tensor(labels[sel]).to(device)
print('train', len(TR[2]), '| test', len(TE[2]))

## Input statistics
Events per bin over time (population mean) and per-sample event counts — quantifies the onset burst and intensity variability that the synthetic proxies lacked.

In [ ]:
import matplotlib.pyplot as plt
x0, y0 = next(batches(TR, 256, shuffle=False))
per_bin = x0.sum(dim=(0, 2)) / x0.shape[0]
per_sample = x0.sum(dim=(1, 2))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(per_bin.numpy()); ax[0].set_xlabel('time bin'); ax[0].set_ylabel('events/bin (mean)')
ax[0].set_title('Population event rate over time')
ax[1].hist(per_sample.numpy(), bins=30); ax[1].set_xlabel('events per sample'); ax[1].set_title('Intensity variability')
plt.tight_layout(); plt.savefig(os.path.join(OUT, 'fig_input_stats.png'), dpi=300); plt.show()
print('peak/median events per bin:', float(per_bin.max()), '/', float(per_bin.median()))

In [ ]:
class Triangle(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x); return (x >= 0).float()
    @staticmethod
    def backward(ctx, g):
        (x,) = ctx.saved_tensors
        return g * torch.clamp(1.0 - x.abs(), min=0.0)
spike_fn = Triangle.apply
def decay(tau): return math.exp(-1.0 / tau)

class PGLIFCell(nn.Module):
    def __init__(self, N, beta=1.0, kappa0=1.0, tau_p=None, theta_d=1.0, tref_p=10, telemetry=False):
        super().__init__(); self.N = N
        self.am = decay(20); self.ad = decay(20); self.aa = decay(200)
        tau_p = tau_p or T_BINS / 2
        ap0 = decay(tau_p)
        self.ap_logit = nn.Parameter(torch.full((N,), math.log(ap0 / (1 - ap0))))
        self.kappa = nn.Parameter(torch.full((N,), float(kappa0)))
        self.beta, self.th, self.th_d, self.P0, self.tref_p = beta, 1.0, theta_d, 1.0, tref_p
        self.telemetry = telemetry
    def init(self, B, dev):
        z = lambda: torch.zeros(B, self.N, device=dev)
        self.vs, self.vd, self.p, self.a = z(), z(), z(), z()
        self.rp = torch.zeros(B, self.N, device=dev)
        if self.telemetry:
            self.stats = {k: 0.0 for k in ['p_mean','p_max','ed_rate','s_rate','a_mean','th_mean',
                                            'live_d','live_s']}; self.tsteps = 0
    def forward(self, I_ff, I_rec):
        self.vd = self.ad * self.vd + I_ff
        ed = spike_fn(self.vd - self.th_d) * (self.rp == 0).float()
        self.rp = torch.clamp(self.rp - 1, min=0) + ed.detach() * self.tref_p
        self.p = torch.sigmoid(self.ap_logit) * self.p + self.P0 * ed
        self.vs = self.am * self.vs + I_ff + I_rec + self.kappa * self.p
        th = self.th + self.beta * self.a
        s = spike_fn(self.vs - th)
        if self.telemetry:
            with torch.no_grad():
                st = self.stats
                st['p_mean'] += self.p.mean().item(); st['p_max'] = max(st['p_max'], self.p.max().item())
                st['ed_rate'] += ed.mean().item(); st['s_rate'] += s.mean().item()
                st['a_mean'] += self.a.mean().item(); st['th_mean'] += th.mean().item()
                st['live_d'] += ((self.vd - self.th_d).abs() < 1).float().mean().item()
                st['live_s'] += ((self.vs - th).abs() < 1).float().mean().item()
                self.tsteps += 1
        self.vs = self.vs - s.detach() * th.detach()
        self.a = self.aa * self.a + s.detach()
        return s
    def epoch_stats(self):
        d = {k: (v / self.tsteps if k != 'p_max' else v) for k, v in self.stats.items()}
        return d

class RecSNN(nn.Module):
    def __init__(self, **cell_kw):
        super().__init__()
        self.w_in = nn.Linear(N_IN, HIDDEN); self.w_rec = nn.Linear(HIDDEN, HIDDEN, bias=False)
        self.cell = PGLIFCell(HIDDEN, **cell_kw); self.w_out = nn.Linear(HIDDEN, N_OUT)
        self.a_out = decay(20); nn.init.orthogonal_(self.w_rec.weight)
    def forward(self, x):
        B, T, _ = x.shape; self.cell.init(B, x.device)
        s = torch.zeros(B, HIDDEN, device=x.device)
        out = torch.zeros(B, N_OUT, device=x.device); vo = torch.zeros(B, N_OUT, device=x.device)
        for t in range(T):
            s = self.cell(self.w_in(x[:, t]), self.w_rec(s))
            vo = self.a_out * vo + self.w_out(s); out = out + vo
        return out

def accuracy(model, split, device, limit=1024):
    model.eval(); correct = tot = 0
    with torch.no_grad():
        for x, y in batches(split, 256, shuffle=False, device=device):
            out = model(x); correct += (out.argmax(1) == y).sum().item(); tot += len(y)
            if tot >= limit: break
    return correct / tot

## Part A — telemetry run (exact v2 configuration)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0); np.random.seed(0)
model = RecSNN(telemetry=True).to(device)
opt = torch.optim.Adam(model.parameters(), lr=LR); crit = nn.CrossEntropyLoss()
groups = {'w_in': model.w_in.parameters, 'w_rec': model.w_rec.parameters,
          'w_out': model.w_out.parameters,
          'kappa': (lambda: [model.cell.kappa]), 'ap_logit': (lambda: [model.cell.ap_logit])}
DIAG = []
for ep in range(5):
    model.train(); gnorms = {k: 0.0 for k in groups}; nb_ = 0
    for x, y in batches(TR, BATCH, shuffle=True, device=device):
        opt.zero_grad(); out = model(x); crit(out, y).backward()
        if nb_ % 40 == 0:
            for k, ps in groups.items():
                gnorms[k] += float(sum((p.grad.detach().norm().item()**2 for p in ps() if p.grad is not None)) ** 0.5)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0); opt.step(); nb_ += 1
    st = model.cell.epoch_stats()
    rec = {'epoch': ep, 'train_acc': accuracy(model, TR, device), 'test_acc': accuracy(model, TE, device),
           **st, 'grad_norms': {k: v / max(1, nb_ // 40) for k, v in gnorms.items()},
           'kappa_mean': float(model.cell.kappa.mean()),
           'tau_p_mean': float((-1 / torch.log(torch.sigmoid(model.cell.ap_logit))).mean())}
    DIAG.append(rec); print(json.dumps(rec))
json.dump(DIAG, open(os.path.join(OUT, 'telemetry.json'), 'w'), indent=2)

In [ ]:
# state distributions after training (one test batch)
model.eval(); x, y = next(batches(TE, 128, shuffle=False, device=device))
with torch.no_grad():
    B, T, _ = x.shape; model.cell.init(B, device); s = torch.zeros(B, HIDDEN, device=device)
    vd_th, vs_th, ps = [], [], []
    for t in range(T):
        iff = model.w_in(x[:, t]); irec = model.w_rec(s)
        model.cell.telemetry = False
        vd_prev = model.cell.vd
        s = model.cell(iff, irec)
        vd_th.append((model.cell.vd - model.cell.th_d).flatten().cpu())
        vs_th.append((model.cell.vs).flatten().cpu()); ps.append(model.cell.p.flatten().cpu())
fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))
ax[0].hist(torch.cat(vd_th).numpy(), bins=80); ax[0].set_title('Vd - theta_d (surrogate live in [-1,1])')
ax[1].hist(torch.cat(vs_th).numpy(), bins=80); ax[1].set_title('Vs after reset')
ax[2].hist(torch.cat(ps).numpy(), bins=80); ax[2].set_title('plateau p')
plt.tight_layout(); plt.savefig(os.path.join(OUT, 'fig_state_hists.png'), dpi=300); plt.show()

## Part B — bracketing sweep (5 epochs each)

In [ ]:
def short_train(tag, epochs=5, **cell_kw):
    torch.manual_seed(0); np.random.seed(0)
    m = RecSNN(**cell_kw).to(device)
    o = torch.optim.Adam(m.parameters(), lr=LR); cr = nn.CrossEntropyLoss()
    hist = []
    for ep in range(epochs):
        m.train()
        for x, y in batches(TR, BATCH, shuffle=True, device=device):
            o.zero_grad(); cr(m(x), y).backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 5.0); o.step()
        hist.append({'epoch': ep, 'train_acc': accuracy(m, TR, device), 'test_acc': accuracy(m, TE, device)})
        print(tag, hist[-1])
    return {'tag': tag, 'config': {k: str(v) for k, v in cell_kw.items()}, 'history': hist}

SWEEP = []
SWEEP.append(short_train('base (as run)'))
SWEEP.append(short_train('beta0 (no adaptation)', beta=0.0))
SWEEP.append(short_train('kappa 0.25', kappa0=0.25))
SWEEP.append(short_train('tau_p 15', tau_p=15))
SWEEP.append(short_train('theta_d 2.0', theta_d=2.0))
json.dump(SWEEP, open(os.path.join(OUT, 'sweep.json'), 'w'), indent=2)
print()
for r in SWEEP:
    best = max(h['test_acc'] for h in r['history'])
    tra = r['history'][-1]['train_acc']
    print(f"{r['tag']:24s} final train {tra:.3f}  best test {best:.3f}")

### Reading the results
- **Train ≈ test ≈ low** for the base run → optimization failure; look at `live_d`/`live_s` (surrogate liveness), `grad_norms`, and `p_max` in the telemetry to see which mechanism (dead surrogate zones, plateau saturation, tiny w_in gradients).
- **Train high, test low** → generalization failure specific to real SHD's speaker split.
- If **beta0** rescues accuracy → the adaptation term interacts destructively with the plateau drive on real input statistics (consistent with the offline onset-burst finding and with TC-LIF, which has no adaptation, being the strongest baseline).
- If **kappa 0.25** or **tau_p 15** rescues → the plateau drive is mis-scaled at initialization for real SHD.

Upload this executed notebook back and the fix goes into P1 v3.